In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    OneHotEncoder
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import (
    LogisticRegression,
    LinearRegression
)

from sklearn.tree import (
    DecisionTreeClassifier,
    DecisionTreeRegressor
)

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor
)

from sklearn.metrics import (
    accuracy_score,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

In [ ]:
import tkinter as tk
from tkinter import filedialog

# Create and hide the main tkinter window
root = tk.Tk()
root.withdraw()

# Open file picker
file_path = filedialog.askopenfilename(
    title="Select your CSV dataset",
    filetypes=[
        ("CSV files", "*.csv"),
        ("All files", "*.*")
    ]
)

# Load the selected dataset
if file_path:

    df = pd.read_csv(file_path)

    print("Dataset loaded successfully!")
    print("File:", file_path)
    print("Shape:", df.shape)

else:

    print("No file selected.")

In [ ]:
print("First 10 rows of the dataset:")

display(df.head(10))

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
missing_values = df.isnull().sum()

missing_table = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": missing_values.values
})

display(missing_table)

In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

In [ ]:
numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = df.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical columns:")
print(numerical_columns)

print("\nCategorical columns:")
print(categorical_columns)

In [ ]:
print("Statistical Summary:")

display(
    df.describe(include="all").transpose()
)

In [ ]:
def clean_dataset(data):

    cleaned_data = data.copy()

    # Remove duplicate rows
    cleaned_data = (
        cleaned_data
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Handle missing values
    for column in cleaned_data.columns:

        if cleaned_data[column].isnull().sum() == 0:
            continue

        # Numerical column
        if pd.api.types.is_numeric_dtype(
            cleaned_data[column]
        ):

            median_value = (
                cleaned_data[column]
                .median()
            )

            cleaned_data[column] = (
                cleaned_data[column]
                .fillna(median_value)
            )

        # Categorical column
        else:

            mode_value = (
                cleaned_data[column]
                .mode()
            )

            if len(mode_value) > 0:

                cleaned_data[column] = (
                    cleaned_data[column]
                    .fillna(mode_value.iloc[0])
                )

            else:

                cleaned_data[column] = (
                    cleaned_data[column]
                    .fillna("Unknown")
                )

    return cleaned_data

In [ ]:
cleaned_df = clean_dataset(df)

print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)

print(
    "\nRemaining missing values:",
    cleaned_df.isnull().sum().sum()
)

print(
    "Remaining duplicate rows:",
    cleaned_df.duplicated().sum()
)

In [ ]:
display(cleaned_df.head(10))

In [ ]:
numerical_columns_cleaned = cleaned_df.select_dtypes(
    include=np.number
).columns.tolist()

if len(numerical_columns_cleaned) > 0:

    for column in numerical_columns_cleaned:

        plt.figure(figsize=(8, 4))

        sns.histplot(
            cleaned_df[column],
            kde=True
        )

        plt.title(
            f"Distribution of {column}"
        )

        plt.xlabel(column)
        plt.ylabel("Frequency")

        plt.show()

else:

    print("No numerical columns found.")

In [ ]:
categorical_columns_cleaned = cleaned_df.select_dtypes(
    exclude=np.number
).columns.tolist()

if len(categorical_columns_cleaned) > 0:

    for column in categorical_columns_cleaned:

        plt.figure(figsize=(8, 4))

        cleaned_df[column].value_counts().head(15).plot(
            kind="bar"
        )

        plt.title(
            f"Distribution of {column}"
        )

        plt.xlabel(column)
        plt.ylabel("Count")

        plt.xticks(rotation=45)

        plt.show()

else:

    print("No categorical columns found.")

In [ ]:
numeric_data = cleaned_df.select_dtypes(
    include=np.number
)

if numeric_data.shape[1] >= 2:

    correlation = numeric_data.corr()

    plt.figure(figsize=(10, 6))

    sns.heatmap(
        correlation,
        annot=True,
        cmap="coolwarm"
    )

    plt.title("Correlation Matrix")

    plt.show()

else:

    print(
        "At least two numerical columns are required."
    )

In [ ]:
if len(numerical_columns_cleaned) > 0:

    for column in numerical_columns_cleaned:

        plt.figure(figsize=(8, 3))

        sns.boxplot(
            x=cleaned_df[column]
        )

        plt.title(
            f"Outliers in {column}"
        )

        plt.show()

else:

    print("No numerical columns available.")

In [ ]:
def detect_problem_type(target):

    # Categorical target
    if target.dtype == "object":

        return "Classification"

    # Boolean target
    if target.dtype == "bool":

        return "Classification"

    # Small number of unique values
    if target.nunique() <= 10:

        return "Classification"

    # Otherwise regression
    return "Regression"

In [ ]:
from IPython.display import display
import ipywidgets as widgets

print("Available columns:")

for i, column in enumerate(cleaned_df.columns):

    print(i, ":", column)

target_dropdown = widgets.Dropdown(
    options=cleaned_df.columns.tolist(),
    description="Target:",
    style={"description_width": "initial"}
)

display(target_dropdown)

In [ ]:
target_column = target_dropdown.value

target = cleaned_df[target_column]

problem_type = detect_problem_type(target)

print("Selected target column:", target_column)

print(
    "Detected problem type:",
    problem_type
)

In [ ]:
def prepare_data(data, target_column):

    X = data.drop(
        columns=[target_column]
    )

    y = data[target_column]

    numerical_columns = X.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_columns = X.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    numerical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]
    )

    transformers = []

    if len(numerical_columns) > 0:

        transformers.append(
            (
                "numerical",
                numerical_pipeline,
                numerical_columns
            )
        )

    if len(categorical_columns) > 0:

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns
            )
        )

    preprocessor = ColumnTransformer(
        transformers=transformers
    )

    return X, y, preprocessor

In [ ]:
X, y, preprocessor = prepare_data(
    cleaned_df,
    target_column
)

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(target_column)

In [ ]:
if problem_type == "Classification":

    label_encoder = LabelEncoder()

    y_encoded = label_encoder.fit_transform(
        y.astype(str)
    )

    try:

        X_train, X_test, y_train, y_test = (
            train_test_split(
                X,
                y_encoded,
                test_size=0.2,
                random_state=42,
                stratify=y_encoded
            )
        )

    except ValueError:

        print(
            "Stratified split was not possible."
        )

        X_train, X_test, y_train, y_test = (
            train_test_split(
                X,
                y_encoded,
                test_size=0.2,
                random_state=42
            )
        )

    models = {

        "Logistic Regression":
            LogisticRegression(
                max_iter=1000
            ),

        "Decision Tree":
            DecisionTreeClassifier(
                random_state=42
            ),

        "Random Forest":
            RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )
    }

    results = []

    trained_models = {}

    for name, model in models.items():

        pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    preprocessor
                ),
                (
                    "model",
                    model
                )
            ]
        )

        pipeline.fit(
            X_train,
            y_train
        )

        predictions = pipeline.predict(
            X_test
        )

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        results.append({

            "Model": name,

            "Accuracy": round(
                accuracy * 100,
                2
            )
        })

        trained_models[name] = pipeline

    results_df = pd.DataFrame(results)

    results_df = results_df.sort_values(
        by="Accuracy",
        ascending=False
    ).reset_index(drop=True)

    display(results_df)

    best_model_name = (
        results_df.iloc[0]["Model"]
    )

    print(
        "Best model:",
        best_model_name
    )

In [ ]:
if problem_type == "Regression":

    X_train, X_test, y_train, y_test = (
        train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42
        )
    )

    models = {

        "Linear Regression":
            LinearRegression(),

        "Decision Tree":
            DecisionTreeRegressor(
                random_state=42
            ),

        "Random Forest":
            RandomForestRegressor(
                n_estimators=100,
                random_state=42
            )
    }

    results = []

    trained_models = {}

    for name, model in models.items():

        pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    preprocessor
                ),
                (
                    "model",
                    model
                )
            ]
        )

        pipeline.fit(
            X_train,
            y_train
        )

        predictions = pipeline.predict(
            X_test
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                predictions
            )
        )

        mae = mean_absolute_error(
            y_test,
            predictions
        )

        r2 = r2_score(
            y_test,
            predictions
        )

        results.append({

            "Model": name,

            "RMSE": round(
                rmse,
                2
            ),

            "MAE": round(
                mae,
                2
            ),

            "R2 Score": round(
                r2,
                3
            )
        })

        trained_models[name] = pipeline

    results_df = pd.DataFrame(
        results
    )

    results_df = results_df.sort_values(
        by="R2 Score",
        ascending=False
    ).reset_index(drop=True)

    display(results_df)

    best_model_name = (
        results_df.iloc[0]["Model"]
    )

    print(
        "Best model:",
        best_model_name
    )

In [ ]:
if problem_type == "Classification":

    plt.figure(figsize=(8, 5))

    plt.bar(
        results_df["Model"],
        results_df["Accuracy"]
    )

    plt.title(
        "Classification Model Comparison"
    )

    plt.xlabel("Model")
    plt.ylabel("Accuracy (%)")

    plt.xticks(rotation=20)

    plt.show()

else:

    plt.figure(figsize=(8, 5))

    plt.bar(
        results_df["Model"],
        results_df["R2 Score"]
    )

    plt.title(
        "Regression Model Comparison"
    )

    plt.xlabel("Model")
    plt.ylabel("R2 Score")

    plt.xticks(rotation=20)

    plt.show()

In [ ]:
cleaned_df.to_csv(
    "cleaned_data.csv",
    index=False
)

print(
    "Cleaned dataset saved successfully!"
)

print(
    "File name: cleaned_data.csv"
)

In [ ]:
print("=" * 50)
print("AUTOMATED DATA SCIENCE SYSTEM")
print("=" * 50)

print(
    "Original dataset shape:",
    df.shape
)

print(
    "Cleaned dataset shape:",
    cleaned_df.shape
)

print(
    "Target column:",
    target_column
)

print(
    "Problem type:",
    problem_type
)

print(
    "Best model:",
    best_model_name
)

print("=" * 50)